# Validación del Pipeline de Datos (CIC-IDS2017)

**Autor:** Daniel Gomollón Embid  
**TFG:** Sistema de Detección de Intrusiones Adversario (CNN vs Transformer)

---

### 🎯 Objetivo de este Notebook
Antes de entrenar cualquier modelo de Deep Learning, es crítico validar que los datos fluyen correctamente desde el almacenamiento en crudo hasta los tensores de PyTorch. En este script verificaremos:

1.  **Conectividad:** Que Google Colab puede acceder a nuestra librería local `src`.
2.  **Integridad:** Que no hay valores nulos (`NaN`) ni infinitos.
3.  **Taxonomía:** Que las etiquetas se han agrupado correctamente en las **7 Familias** definidas.
4.  **Escalado:** Que todos los valores numéricos están normalizados entre `[0, 1]` (Requisito indispensable para ataques FGSM/PGD).
5.  **Dimensiones:** Confirmar el `input_shape` para diseñar la primera capa de la CNN.

#**Configuración del Entorno y Montaje de Drive**

In [ ]:

import sys
import os
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np
import pandas as pd

# Configuración de gráficos estilo "Paper Científico"
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Montar Google Drive
print("[-] Montando Google Drive...")
drive.mount('/content/drive')

# 2. Definir ruta del proyecto (¡AJUSTA ESTO SI TU CARPETA SE LLAMA DIFERENTE!)
PROJECT_PATH = '/content/drive/MyDrive/Codigo_Proyecto'

# 3. Verificación de seguridad
if not os.path.exists(PROJECT_PATH):
    print(f"ERROR: La ruta {PROJECT_PATH} no existe. Verifica el nombre en Drive.")
else:
    print(f"Ruta encontrada: {PROJECT_PATH}")
    # Añadir al path de Python para importar 'src'
    if PROJECT_PATH not in sys.path:
        sys.path.append(PROJECT_PATH)
    os.chdir(PROJECT_PATH)
    print("Directorio de trabajo actualizado.")

#**Importación de Módulos del Proyecto (src)**

- Probar que nuestra arquitectura de carpetas (src/__init__.py) funciona.
- Verificar que hay aceleración por hardware, al menos CPU o GPU para entrenar modelos

In [ ]:
print("[-] Importando módulos personalizados...")

try:
    from src.data_pipeline_extended import get_dataloaders, ATTACK_MAPPING
    from src.config import Config

    # Ahora accedemos a DEVICE y DATA_RAW_PATH como atributos de la clase Config
    DEVICE = Config.DEVICE
    DATA_RAW_PATH = Config.DATA_RAW_PATH

    print(f"Módulos importados correctamente.")
    print(f"Dispositivo de aceleración detectado: {DEVICE}")
    print(f"Ruta de datos configurada: {DATA_RAW_PATH}")

except ImportError as e:
    print(f"\nERROR DE IMPORTACIÓN! {e}")

#**Ejecución del Pipeline ETL**

Al llamar a `get_dataloaders()`, se dispara automáticamente el proceso definido en `src/data_pipeline.py`:
1.  **Carga** el CSV crudo.
2.  **Limpia** nulos e infinitos.
3.  **Agrupa** las etiquetas en las 7 familias (DoS, PortScan, etc.).
4.  **Escala** los datos usando `MinMaxScaler` (ajustado solo en Train).
5.  **Empaqueta** todo en `DataLoaders` de PyTorch listos para la GPU.

In [ ]:
# Cargamos solo la primera fila para ver las cabeceras rápido
print(f"[-] Inspeccionando cabeceras de: {DATA_RAW_PATH}")
df_test = pd.read_csv(DATA_RAW_PATH, nrows=1)

print("\n🔍 LISTA REAL DE COLUMNAS:")
print(df_test.columns.tolist())

In [ ]:
# Ejecución del Pipeline (Puede tardar unos segundos la primera vez)
print("[-] Procesando Dataset CIC-IDS2017...")

try:
    train_loader, val_loader, test_loader = get_dataloaders()
    print("\nDataLoaders creados exitosamente.")
    print(f"   - Lotes de Entrenamiento: {len(train_loader)}")
    print(f"   - Lotes de Validación:    {len(val_loader)}")
    print(f"   - Lotes de Test:          {len(test_loader)}")
except Exception as e:
    print(f"Error en el Pipeline: {e}")
    # Tip para depurar:
    import traceback
    traceback.print_exc()

#**Inspección de Tensores**

- Aquí validamos matemáticamente lo que entra a la red

In [ ]:
# Extraemos el primer batch para "hacerle la autopsia"
X_batch, y_batch = next(iter(train_loader))

print(f"--- Dimensiones y Tipos ---")
print(f"Tensor de Entrada (X): {X_batch.shape} | Tipo: {X_batch.dtype}")
print(f"Tensor de Etiquetas (y): {y_batch.shape}  | Tipo: {y_batch.dtype}")

# VALIDACIÓN CRÍTICA PARA CNN
batch_size, num_features = X_batch.shape
print(f"\nCONCLUSIÓN PARA EL MODELO:")
print(f"   > El 'input_shape' para la CNN será: {num_features}")
print(f"   > La CNN deberá hacer un '.unsqueeze(1)' internamente para convertir")
print(f"     [Batch, {num_features}] -> [Batch, 1, {num_features}]")

# VALIDACIÓN CRÍTICA PARA ATAQUES ADVERSARIOS
min_val = X_batch.min().item()
max_val = X_batch.max().item()

print(f"\n--- Validación de Escalado (MinMax) ---")
print(f"Valor Mínimo en el batch: {min_val:.4f}")
print(f"Valor Máximo en el batch: {max_val:.4f}")

if -0.01 <= min_val and max_val <= 1.01:
    print("CORRECTO: Los datos están normalizados en [0, 1].")
    print("   Los ataque FGSM y PGD funcionarán correctamente.")
else:
    print("PELIGRO: Los datos NO están en [0, 1]. Revisar 'data_pipeline.py'.")

#**Visualización de la Distribución de Clases**

In [ ]:
# Recuperamos los nombres de las clases del diccionario en config/pipeline
idx_to_class = {v: k for k, v in ATTACK_MAPPING.items()}

# Contamos las clases en este batch (es una muestra representativa)
etiquetas_unicas, conteos = torch.unique(y_batch, return_counts=True)

# Convertimos a listas para graficar
nombres_clases = [idx_to_class[i.item()] for i in etiquetas_unicas]
conteos_lista = conteos.tolist()

# Crear Gráfico
plt.figure(figsize=(12, 6))
barplot = sns.barplot(x=nombres_clases, y=conteos_lista, palette="viridis", hue=nombres_clases, legend=False)

plt.title("Distribución de Familias de Ataques (Muestra de un Batch)", fontsize=15, fontweight='bold')
plt.xlabel("Familia de Ataque", fontsize=12)
plt.ylabel("Número de Muestras", fontsize=12)
plt.xticks(rotation=45)

# Añadir etiquetas de valor encima de las barras
for p in barplot.patches:
    barplot.annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha = 'center', va = 'center',
                     xytext = (0, 9),
                     textcoords = 'offset points')

plt.tight_layout()
plt.show()

print("\nNOTA: Es normal que 'Normal Traffic' domine. Es cerca del 80% del dataset. Si usamos 'WeightedRandomSampler' o 'Class Weights'")
print("        en el entrenamiento, el modelo aprenderá a no ignorar las barras pequeñas.")